# P-ML8 — Volume Feature Engineering

**Motivation (F14 / P-ML7):** The P-ML7 ensemble (16 features, Sharpe +1.261) closes 73% of the
gap to Buy & Hold (+1.379) by adding momentum features. One notable remaining gap: the model
cannot distinguish *broad-conviction* moves (high volume) from *low-participation* drifts
where reversal risk is high.

**Two theoretical frameworks motivate volume features:**

1. **SENTIMENT THEORY:** Volume validates price moves. High volume on an up-day signals broad
   conviction; high volume on a down-day signals broad panic/distribution; low volume on any
   move signals lack of conviction and reversal likelihood.

2. **INSTITUTIONAL PARTICIPATION THEORY:** BTC has seen growing institutional participation
   since ~2020 (Grayscale, MicroStrategy), accelerating with US spot ETF approval in January
   2024 (BlackRock, Fidelity). Institutions trade in size, leaving footprints: steady consistent
   volume with upward bias (accumulation) and periodic block-trade spikes.

**Hypothesis (H_vol):** Volume-based features add directional and conviction signals not captured
by price-based oscillators alone. If the institutional theory holds, volume features should show
**increasing predictive power post-2020** as institutional participation grows.

**Method:** IC screen on 8 volume candidates (stratified by regime AND institutional era)
→ select features with |IC_bull| > 0.01 → re-run P-ML7 `RegimeEnsemble` on augmented
feature set FEATURES_V3 → compare metrics.

## §1 — Config

In [1]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         9,
})

# ── Dataset (identical to P-ML5 / P-ML7) ──────────────────────────────────────
SYMBOL     = "BTC/USDT"
SINCE      = "2019-01-01"
UNTIL      = "2025-01-01"
HORIZON    = 1
N_SPLITS   = 5
TRAIN_FRAC = 0.6
PURGE      = 1
LONG_MA    = 200
ADX_THRESH = 25.0
MIN_BULL_BARS = 30

# ── Feature sets from prior experiments ───────────────────────────────────────
FEATURES = [
    "bar_ret", "bb_zscore", "rsi", "macd_hist_norm", "atr_pct",
    "bb_width", "upper_wick", "lower_wick", "hl_range",
    "vol_log_chg", "di_diff", "adx",
]  # original 12

FEATURES_V2 = FEATURES + [
    "ret_5", "ret_20", "mom_zscore_20", "ret_5_minus_20",
]  # P-ML7 16-feature set

# ── Volume feature selection ───────────────────────────────────────────────────
IC_THRESHOLD = 0.01       # minimum |IC_bull| to include a feature

# ── Institutional era definitions ──────────────────────────────────────────────
# Era 1: 2019–2020  — retail-dominated; limited institutional presence
# Era 2: 2021       — first institutional wave (Grayscale, MicroStrategy, Tesla)
# Era 3: 2022–2023  — bear market + FTX collapse; institutional re-evaluation
# Era 4: 2024       — US spot ETF approval (Jan 2024, BlackRock/Fidelity)
ERA_LABELS = {
    "Era1_Retail (2019-2020)": ("2019-01-01", "2021-01-01"),
    "Era2_Instit1 (2021)":     ("2021-01-01", "2022-01-01"),
    "Era3_Bear (2022-2023)":   ("2022-01-01", "2024-01-01"),
    "Era4_ETF (2024)": ("2024-01-01", "2025-01-01"),
}

# ── Reference baselines ────────────────────────────────────────────────────────
P_ML5_ICS = [0.0612, -0.0536, 0.1295, 0.0462, 0.0861]
BUY_HOLD  = {"return": 2.996, "sharpe": 1.379, "maxdd": -0.354}
P_ML5     = {"return": 6.302, "sharpe": 0.927, "maxdd": -0.680}
P_ML7     = {"return": 19.976, "sharpe": 1.261, "maxdd": -0.773}
P_ML7_ICS = [0.0721, 0.0091, 0.1283, 0.0537, 0.1069]

print(f"Dataset:   {SINCE} → {UNTIL} | 1d | horizon={HORIZON}")
print(f"Walk-forward: {N_SPLITS} folds, train_frac={TRAIN_FRAC}, purge={PURGE}")
print(f"IC threshold: |IC_bull| > {IC_THRESHOLD}")
print(f"\nP-ML7 reference: Sharpe={P_ML7['sharpe']:+.3f}, Return={P_ML7['return']*100:+.1f}%, MaxDD={P_ML7['maxdd']*100:.1f}%")
print(f"\nInstitutional era periods:")
for name, (s, e) in ERA_LABELS.items():
    print(f"  {name}: {s} → {e}")

Dataset:   2019-01-01 → 2025-01-01 | 1d | horizon=1
Walk-forward: 5 folds, train_frac=0.6, purge=1
IC threshold: |IC_bull| > 0.01

P-ML7 reference: Sharpe=+1.261, Return=+1997.6%, MaxDD=-77.3%

Institutional era periods:
  Era1_Retail (2019-2020): 2019-01-01 → 2021-01-01
  Era2_Instit1 (2021): 2021-01-01 → 2022-01-01
  Era3_Bear (2022-2023): 2022-01-01 → 2024-01-01
  Era4_ETF (2024): 2024-01-01 → 2025-01-01


## §2 — Data Loading + Volume Landscape

Load the 6yr dataset, construct volume features, and visualise how BTC trading volume
has evolved across institutional eras.

In [2]:
from data.fetch import fetch_ohlcv
from ml.features import build_feature_matrix
from ml.features.momentum import build_momentum_features
from ml.features.volume import build_volume_features
from ml.labels import forward_return
from ml.regime import RegimeClassifier
from ml.validation import purged_wf_splits

# Cache-guard (same as P-ML7)
df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d", since=SINCE, until=UNTIL)
if df_raw.index.min() > pd.Timestamp("2020-01-01", tz="UTC"):
    print("Cache missing early data — re-fetching 2019–2025 from exchange...")
    df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d",
                         since=SINCE, until=UNTIL, use_cache=False)
print(f"Loaded {len(df_raw):,} bars  |  {df_raw.index[0].date()} → {df_raw.index[-1].date()}")

# Build all feature groups and label
feats_orig = build_feature_matrix(df_raw)                       # original 12
feats_mom  = build_momentum_features(df_raw)                    # momentum 4
feats_vol  = build_volume_features(df_raw)                      # volume candidates
label      = forward_return(df_raw, horizon=HORIZON)

# Combine and drop NaN (vol warmup is max 30 bars, shorter than mom's 60)
comb = pd.concat([feats_orig, feats_mom, feats_vol, label], axis=1).dropna()

X_v2   = comb[FEATURES_V2]
y_all  = comb[label.name]

# Volume candidate columns
VOL_CANDIDATES = [c for c in feats_vol.columns if c in comb.columns]

# Regime labels
rc  = RegimeClassifier(long_ma=LONG_MA, adx_thresh=ADX_THRESH)
reg = rc.transform(df_raw).reindex(comb.index)
reg["regime"] = reg["regime"].fillna("ranging")

bar_ret_daily = np.log(df_raw["close"] / df_raw["close"].shift(1)).reindex(comb.index)
splits = list(purged_wf_splits(len(comb), N_SPLITS, TRAIN_FRAC, purge_bars=PURGE))

print(f"\n{len(comb):,} usable bars | {comb.index[0].date()} → {comb.index[-1].date()}")
print(f"Volume candidates: {VOL_CANDIDATES}")

Cache missing early data — re-fetching 2019–2025 from exchange...


Loaded 2,193 bars  |  2019-01-01 → 2025-01-01

2,132 usable bars | 2019-03-02 → 2024-12-31
Volume candidates: ['vol_log_ratio_7d', 'vol_log_ratio_14d', 'vol_log_ratio_30d', 'vol_cv_14d', 'vol_zscore_30d', 'vol_trend_7_14', 'vol_signed_ratio_7d', 'vol_signed_ratio_14d', 'vol_price_corr_14d']


In [3]:
# ── Volume landscape: how has BTC volume evolved across institutional eras? ────
vol_series   = df_raw["volume"].reindex(comb.index)
vol_ma_30    = vol_series.rolling(30).mean()
close_series = df_raw["close"].reindex(comb.index)

era_colors = {"Era1_Retail (2019-2020)": "#95a5a6",
              "Era2_Instit1 (2021)":     "#3498db",
              "Era3_Bear (2022-2023)":   "#e74c3c",
              "Era4_ETF (2024)":         "#2ecc71"}

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Price
axes[0].semilogy(close_series.index, close_series.values,
                 color="black", linewidth=1.0, label="BTC price (log scale)")
axes[0].set_ylabel("Price (USD, log)")
axes[0].set_title("BTC/USDT — Price & Volume by Institutional Era (2019–2024)")

# Volume bars colored by era
for era_name, (s, e) in ERA_LABELS.items():
    mask = (vol_series.index >= pd.Timestamp(s, tz="UTC")) & \
           (vol_series.index <  pd.Timestamp(e, tz="UTC"))
    axes[1].bar(vol_series.index[mask],
                vol_series.values[mask],
                color=era_colors[era_name], alpha=0.7,
                label=era_name, width=1)

axes[1].plot(vol_ma_30.index, vol_ma_30.values,
             color="black", linewidth=1.2, linestyle="--", label="30d MA")
axes[1].set_ylabel("Volume (BTC)")
axes[1].legend(fontsize=7, ncol=3)

plt.tight_layout()
plt.show()

# ── Era-level volume statistics ───────────────────────────────────────────────
print(f"\n{'Era':<30} {'Bars':>5} {'MeanVol':>11} {'StdVol':>10} {'CV':>7} {'VolGrowth%':>11}")
print("-" * 70)
era_stats = {}
prev_mean = None
for era_name, (s, e) in ERA_LABELS.items():
    mask = (vol_series.index >= pd.Timestamp(s, tz="UTC")) & \
           (vol_series.index <  pd.Timestamp(e, tz="UTC"))
    v = vol_series[mask]
    m, sd = v.mean(), v.std()
    cv  = sd / m if m > 0 else float("nan")
    growth = (m / prev_mean - 1) * 100 if prev_mean else float("nan")
    era_stats[era_name] = {"mean": m, "std": sd, "cv": cv, "mask": mask}
    g_str = f"{growth:>+9.1f}%" if not np.isnan(growth) else "      —"
    print(f"  {era_name:<28} {mask.sum():>5} {m:>11,.0f} {sd:>10,.0f} "
          f"{cv:>7.3f} {g_str:>11}")
    prev_mean = m


Era                             Bars     MeanVol     StdVol      CV  VolGrowth%
----------------------------------------------------------------------
  Era1_Retail (2019-2020)        671      40,733     29,075   0.714           —
  Era2_Instit1 (2021)            365      17,683     11,004   0.622      -56.6%
  Era3_Bear (2022-2023)          730      10,217      7,648   0.749      -42.2%
  Era4_ETF (2024)                366      10,899      7,414   0.680       +6.7%


## §3 — IC Screen: Volume Features vs Forward Return

For each volume feature, compute Spearman IC stratified by:
- **Regime**: all bars / bull / non-bull
- **Institutional era**: retail (2019-2020), first institutional wave (2021),
  bear+FTX (2022-2023), ETF era (2024)

**Institutional theory prediction:** If volume reflects institutional participation,
`vol_zscore_30d`, `vol_cv_14d`, and `vol_signed_ratio` should show *increasing*
IC magnitude from Era 1 → Era 4 as institutional share grows.

**Selection rule:** include if |IC_bull| > 0.01. Also check collinearity with FEATURES_V2.

In [4]:
bull_mask    = (reg["regime"] == "bull").values
nonbull_mask = ~bull_mask

print(f"Regime split: bull={bull_mask.sum()} bars ({bull_mask.mean()*100:.1f}%)  "
      f"non-bull={nonbull_mask.sum()} ({nonbull_mask.mean()*100:.1f}%)")
print()

# ── Main IC table (regime-stratified) ─────────────────────────────────────────
print(f"{'Feature':<22} {'IC_all':>7} {'p_all':>6} {'IC_bull':>8} {'p_b':>6} "
      f"{'IC_nb':>8} {'p_nb':>6} {'Select?':>8}")
print("-" * 80)

ic_results = {}
for feat in VOL_CANDIDATES:
    x = comb[feat].values
    y = y_all.values

    rho_all,  p_all  = stats.spearmanr(x, y)
    rho_bull, p_bull = stats.spearmanr(x[bull_mask],    y[bull_mask])
    rho_nb,   p_nb   = stats.spearmanr(x[nonbull_mask], y[nonbull_mask])

    selected = abs(rho_bull) > IC_THRESHOLD
    ic_results[feat] = {
        "IC_all": rho_all, "p_all": p_all,
        "IC_bull": rho_bull, "p_bull": p_bull,
        "IC_nonbull": rho_nb, "p_nb": p_nb,
        "selected": selected,
    }
    sel_str = "YES ✓" if selected else "no"
    print(f"  {feat:<20} {rho_all:>+7.4f} {p_all:>6.3f} "
          f"{rho_bull:>+8.4f} {p_bull:>6.3f} "
          f"{rho_nb:>+8.4f} {p_nb:>6.3f} {sel_str:>8}")

selected_vol_feats = [f for f, r in ic_results.items() if r["selected"]]
print(f"\nSelected volume features ({len(selected_vol_feats)}): {selected_vol_feats}")

Regime split: bull=686 bars (32.2%)  non-bull=1446 (67.8%)

Feature                 IC_all  p_all  IC_bull    p_b    IC_nb   p_nb  Select?
--------------------------------------------------------------------------------
  vol_log_ratio_7d     +0.0401  0.064  +0.0348  0.362  +0.0432  0.100    YES ✓
  vol_log_ratio_14d    +0.0442  0.041  +0.0355  0.353  +0.0471  0.073    YES ✓
  vol_log_ratio_30d    +0.0479  0.027  +0.0285  0.456  +0.0538  0.041    YES ✓
  vol_cv_14d           -0.0187  0.389  -0.0425  0.267  -0.0071  0.789    YES ✓
  vol_zscore_30d       +0.0339  0.118  +0.0256  0.503  +0.0361  0.171    YES ✓
  vol_trend_7_14       +0.0247  0.253  +0.0045  0.907  +0.0326  0.215       no
  vol_signed_ratio_7d  -0.0014  0.948  -0.0364  0.341  +0.0002  0.994    YES ✓
  vol_signed_ratio_14d +0.0194  0.371  +0.0230  0.547  +0.0011  0.966    YES ✓
  vol_price_corr_14d   +0.0274  0.206  +0.0455  0.234  +0.0101  0.702    YES ✓

Selected volume features (8): ['vol_log_ratio_7d', 'vol_log_ratio_14

In [5]:
# ── Era-stratified IC — does predictive power grow post-2020? ─────────────────
print("Era-stratified IC on ALL bars (tests institutional participation theory)")
print()
era_cols = list(ERA_LABELS.keys())

header = f"{'Feature':<22}" + "".join(f" {e[:18]:>18}" for e in era_cols)
print(header)
print("-" * (22 + 18 * len(era_cols)))

era_ic_table = {}
for feat in VOL_CANDIDATES:
    era_ics = []
    for era_name, (s, e) in ERA_LABELS.items():
        mask = (comb.index >= pd.Timestamp(s, tz="UTC")) & \
               (comb.index <  pd.Timestamp(e, tz="UTC"))
        if mask.sum() < 10:
            era_ics.append(float("nan"))
            continue
        x_era = comb[feat].values[mask]
        y_era = y_all.values[mask]
        rho, _ = stats.spearmanr(x_era, y_era)
        era_ics.append(rho)
    era_ic_table[feat] = era_ics

    row = f"  {feat:<20}" + "".join(
        f" {ic:>+18.4f}" if not np.isnan(ic) else f" {'  nan':>18}" for ic in era_ics
    )
    print(row)

print()
print("Interpretation: if institutional theory holds, IC magnitude for vol_zscore_30d,")
print("vol_cv_14d, vol_signed_ratio should increase from Era1 → Era4.")

Era-stratified IC on ALL bars (tests institutional participation theory)

Feature                Era1_Retail (2019- Era2_Instit1 (2021 Era3_Bear (2022-20    Era4_ETF (2024)
----------------------------------------------------------------------------------------------
  vol_log_ratio_7d                +0.0135            +0.1526            +0.0078            +0.0123
  vol_log_ratio_14d               +0.0282            +0.1172            +0.0256            +0.0044
  vol_log_ratio_30d               +0.0299            +0.1231            +0.0254            +0.0177
  vol_cv_14d                      -0.0741            +0.0285            +0.0628            +0.0295
  vol_zscore_30d                  -0.0134            +0.1327            +0.0278            +0.0122
  vol_trend_7_14                  +0.0296            -0.0403            +0.0541            +0.0304
  vol_signed_ratio_7d             -0.0092            +0.0027            -0.0026            -0.0244
  vol_signed_ratio_14d            +0.03

In [6]:
# ── Collinearity check vs FEATURES_V2 ─────────────────────────────────────────
if selected_vol_feats:
    corr_check = pd.concat([X_v2, comb[selected_vol_feats]], axis=1).corr()
    cross_corr = corr_check.loc[selected_vol_feats, FEATURES_V2]
    print("Max |correlation| between selected volume features and FEATURES_V2:")
    for feat in selected_vol_feats:
        max_corr = cross_corr.loc[feat].abs().max()
        max_feat = cross_corr.loc[feat].abs().idxmax()
        flag = " ← high overlap" if max_corr > 0.8 else ""
        print(f"  {feat:<22} max|r|={max_corr:.3f} with {max_feat}{flag}")

# ── IC bar chart: volume candidates (all bars vs bull bars) ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# All-bar IC
all_s = pd.Series({f: ic_results[f]["IC_all"] for f in VOL_CANDIDATES}).sort_values()
colors_all = ["#2ecc71" if v > 0 else "#e74c3c" for v in all_s.values]
axes[0].barh(all_s.index, all_s.values, color=colors_all, alpha=0.85)
axes[0].axvline(0, color="black", linewidth=0.7)
axes[0].set_title("IC on all bars: volume candidates", fontsize=9)
axes[0].set_xlabel("Spearman IC")

# Bull-bar IC
bull_s = pd.Series({f: ic_results[f]["IC_bull"] for f in VOL_CANDIDATES}).sort_values()
colors_bull = ["#2ecc71" if v > 0 else "#e74c3c" for v in bull_s.values]
axes[1].barh(bull_s.index, bull_s.values, color=colors_bull, alpha=0.85)
axes[1].axvline(0, color="black", linewidth=0.7)
axes[1].axvline( IC_THRESHOLD, color="gray", linewidth=0.8, linestyle="--",
                 label=f"±{IC_THRESHOLD} threshold")
axes[1].axvline(-IC_THRESHOLD, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_title("IC on bull bars: volume candidates", fontsize=9)
axes[1].set_xlabel("Spearman IC")
axes[1].legend(fontsize=8)

plt.suptitle("Spearman IC — volume feature candidates", fontsize=10)
plt.tight_layout()
plt.show()

# ── Era-trend heatmap for institutional theory test ────────────────────────────
era_ic_df = pd.DataFrame(era_ic_table, index=era_cols).T

fig, ax = plt.subplots(figsize=(11, max(3, len(VOL_CANDIDATES) * 0.5 + 1)))
vmax = era_ic_df.abs().max().max()
im   = ax.imshow(era_ic_df.values, cmap="RdYlGn", aspect="auto",
                 vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(era_cols)))
ax.set_xticklabels([e[:22] for e in era_cols], rotation=20, ha="right", fontsize=8)
ax.set_yticks(range(len(VOL_CANDIDATES)))
ax.set_yticklabels(VOL_CANDIDATES, fontsize=8)
# Annotate cells
for i, feat in enumerate(VOL_CANDIDATES):
    for j in range(len(era_cols)):
        v = era_ic_df.iloc[i, j]
        txt = f"{v:+.3f}" if not np.isnan(v) else "nan"
        ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                color="black")
plt.colorbar(im, ax=ax, label="Spearman IC")
ax.set_title("Era-stratified IC — do volume signals grow with institutional adoption?",
             fontsize=9)
plt.tight_layout()
plt.show()

Max |correlation| between selected volume features and FEATURES_V2:
  vol_log_ratio_7d       max|r|=0.656 with vol_log_chg
  vol_log_ratio_14d      max|r|=0.592 with vol_log_chg
  vol_log_ratio_30d      max|r|=0.555 with hl_range
  vol_cv_14d             max|r|=0.333 with ret_20
  vol_zscore_30d         max|r|=0.581 with hl_range
  vol_signed_ratio_7d    max|r|=0.691 with bb_zscore
  vol_signed_ratio_14d   max|r|=0.739 with rsi
  vol_price_corr_14d     max|r|=0.624 with mom_zscore_20


## §4 — Augmented Walk-Forward with RegimeEnsemble

Re-run the P-ML7 walk-forward with `FEATURES_V3 = FEATURES_V2 + selected_vol_feats`.

Primary question: do volume features improve IC or equity on top of the 16-feature P-ML7 baseline?
Secondary: does per-fold IC pattern change (fewer negative folds)?

In [7]:
from ml.models import RegimeEnsemble
from backtesting import compute_metrics

if not selected_vol_feats:
    print("No volume features passed IC screen — FEATURES_V3 == FEATURES_V2.")
    print("Walk-forward will reproduce P-ML7 results exactly.")
    FEATURES_V3 = FEATURES_V2
else:
    FEATURES_V3 = FEATURES_V2 + selected_vol_feats

print(f"FEATURES_V3 ({len(FEATURES_V3)} features): {FEATURES_V3}")
print()

X_v3 = comb[FEATURES_V3]

# ── Walk-forward loop ─────────────────────────────────────────────────────────
fold_results_P8 = []
bull_ics_p8     = []
nb_ics_p8       = []

print(f"{'Fold':<5} {'Period':<35} {'n_train':>8} {'bull_tr':>7} "
      f"{'P7 IC':>8} {'P8 IC':>8} {'Δ IC':>7} {'hit':>6}")
print("-" * 88)

for i, (tr, te) in enumerate(splits):
    ensemble = RegimeEnsemble(min_bull_bars=MIN_BULL_BARS)
    ensemble.fit(X_v3.iloc[tr], y_all.iloc[tr], reg["regime"].iloc[tr])
    preds  = ensemble.predict(X_v3.iloc[te], reg["regime"].iloc[te])
    actual = y_all.iloc[te].values

    rho, pval = stats.spearmanr(preds, actual)
    hit       = (np.sign(preds) == np.sign(actual)).mean()

    bull_te = (reg["regime"].iloc[te] == "bull").values
    nb_te   = ~bull_te

    if bull_te.sum() >= 2:
        if ensemble.has_bull_model:
            pb = ensemble.bull_model.predict(X_v3.iloc[te][bull_te])
        else:
            pb = -ensemble.non_bull_model.predict(X_v3.iloc[te][bull_te])
        bull_ic_p8, _ = stats.spearmanr(pb, actual[bull_te])
    else:
        bull_ic_p8 = float("nan")
    bull_ics_p8.append(bull_ic_p8)

    if nb_te.sum() >= 2:
        pnb = ensemble.non_bull_model.predict(X_v3.iloc[te][nb_te])
        nb_ic_p8, _ = stats.spearmanr(pnb, actual[nb_te])
    else:
        nb_ic_p8 = float("nan")
    nb_ics_p8.append(nb_ic_p8)

    n_bull_tr = int((reg["regime"].iloc[tr] == "bull").sum())
    p7_ic     = P_ML7_ICS[i]
    delta     = rho - p7_ic
    period    = f"{comb.index[te[0]].date()} → {comb.index[te[-1]].date()}"

    print(f"  {i+1:<3} {period:<35} {len(tr):>8} {n_bull_tr:>7} "
          f"{p7_ic:>+8.4f} {rho:>+8.4f} {delta:>+7.4f} {hit:>6.3f}")

    fold_results_P8.append({
        "fold": i + 1, "te": te, "IC": rho, "IC_pval": pval,
        "hit": hit, "preds": preds, "ensemble": ensemble,
    })

ics_P8 = [r["IC"] for r in fold_results_P8]
print()
print(f"P-ML8: Mean IC={np.mean(ics_P8):+.4f}  ICIR={np.mean(ics_P8)/np.std(ics_P8):.3f}")
print(f"P-ML7: Mean IC={np.mean(P_ML7_ICS):+.4f}  ICIR={np.mean(P_ML7_ICS)/np.std(P_ML7_ICS):.3f}  [reference]")

FEATURES_V3 (24 features): ['bar_ret', 'bb_zscore', 'rsi', 'macd_hist_norm', 'atr_pct', 'bb_width', 'upper_wick', 'lower_wick', 'hl_range', 'vol_log_chg', 'di_diff', 'adx', 'ret_5', 'ret_20', 'mom_zscore_20', 'ret_5_minus_20', 'vol_log_ratio_7d', 'vol_log_ratio_14d', 'vol_log_ratio_30d', 'vol_cv_14d', 'vol_zscore_30d', 'vol_signed_ratio_7d', 'vol_signed_ratio_14d', 'vol_price_corr_14d']

Fold  Period                               n_train bull_tr    P7 IC    P8 IC    Δ IC    hit
----------------------------------------------------------------------------------------
  1   2020-02-20 → 2021-02-08                  354      37  +0.0721  -0.0391 -0.1112  0.448


  2   2021-02-09 → 2022-01-29                  531     216  +0.0091  +0.0027 -0.0064  0.501
  3   2022-01-30 → 2023-01-19                  531     253  +0.1283  +0.1434 +0.0151  0.538


  4   2023-01-20 → 2024-01-09                  531      92  +0.0537  +0.0701 +0.0164  0.518
  5   2024-01-10 → 2024-12-29                  531     214  +0.1069  +0.0545 -0.0524  0.518

P-ML8: Mean IC=+0.0463  ICIR=0.747
P-ML7: Mean IC=+0.0740  ICIR=1.779  [reference]


## §5 — Equity Comparison & Finding F15

In [8]:
def build_equity(fold_preds_list, bar_ret_daily, idx):
    pieces, anchor = [], 1.0
    for te, preds in fold_preds_list:
        pos   = np.sign(preds)
        ret   = bar_ret_daily.iloc[te].values
        eq    = np.cumprod(1 + np.roll(pos, 1) * ret)
        eq[0] = 1.0
        s = pd.Series(eq, index=idx[te])
        s = s / s.iloc[0] * anchor
        anchor = float(s.iloc[-1])
        pieces.append(s)
    return pd.concat(pieces)


idx    = comb.index
oos_P8 = build_equity([(r["te"], r["preds"]) for r in fold_results_P8], bar_ret_daily, idx)
bah    = df_raw["close"].reindex(oos_P8.index)
bah    = bah / bah.iloc[0]

m_p8  = compute_metrics(oos_P8)

# ── Equity plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
oos_P8.plot(ax=ax, label=f"P-ML8 (LGBM + momentum + volume [{len(FEATURES_V3)}f])",
            color="#e67e22", linewidth=2.0)
bah.plot(   ax=ax, label="Buy-and-Hold",
            color="black", linewidth=1.0, linestyle=":")
for _, (tr, te) in enumerate(splits):
    ax.axvline(idx[te[0]], color="lightgray", linewidth=0.5, linestyle="--")
ax.axhline(1, color="black", linewidth=0.4)
ax.set_title("OOS Equity — P-ML8 (volume features) vs P-ML7 / Buy-and-Hold", fontsize=11)
ax.set_ylabel("Equity (normalised)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Fold-by-fold IC chart: P-ML7 vs P-ML8 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_SPLITS)
w = 0.35
ax.bar(x - w/2, P_ML7_ICS, w, label="P-ML7 (16 features)",
       color="#9b59b6", alpha=0.85)
ax.bar(x + w/2, ics_P8,    w, label=f"P-ML8 ({len(FEATURES_V3)} features)",
       color="#e67e22", alpha=0.85)
ax.axhline(0,     color="black", linewidth=0.7)
ax.axhline(+0.03, color="gray",  linewidth=0.6, linestyle="--", label="±0.03 target")
ax.axhline(-0.03, color="gray",  linewidth=0.6, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels([f"F{i+1}" for i in range(N_SPLITS)])
ax.set_ylabel("OOS IC")
ax.set_title("OOS IC per fold: P-ML7 (momentum) vs P-ML8 (+volume)", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ── Metrics table ─────────────────────────────────────────────────────────────
print(f"\n{'Strategy':<40} {'Return':>9} {'Sharpe':>8} {'MaxDD':>8}")
print("-" * 68)
rows = [
    ("Buy & Hold",                    BUY_HOLD["return"],     BUY_HOLD["sharpe"],   BUY_HOLD["maxdd"]),
    ("P-ML5 (6yr, 12 feats)",         P_ML5["return"],       P_ML5["sharpe"],      P_ML5["maxdd"]),
    ("P-ML7 (6yr, 16 feats, mom)",    P_ML7["return"],       P_ML7["sharpe"],      P_ML7["maxdd"]),
    (f"P-ML8 (6yr, {len(FEATURES_V3)}f, vol)",
                                       m_p8["total_return"],  m_p8["sharpe_ratio"], m_p8["max_drawdown"]),
]
for name, ret, shr, mdd in rows:
    print(f"  {name:<38} {ret*100:>+8.1f}%  {shr:>+7.3f}  {mdd*100:>7.1f}%")


Strategy                                    Return   Sharpe    MaxDD
--------------------------------------------------------------------
  Buy & Hold                               +299.6%   +1.379    -35.4%
  P-ML5 (6yr, 12 feats)                    +630.2%   +0.927    -68.0%
  P-ML7 (6yr, 16 feats, mom)              +1997.6%   +1.261    -77.3%
  P-ML8 (6yr, 24f, vol)                     -43.2%   +0.180    -91.5%


In [9]:
# ── Finding F15 summary ───────────────────────────────────────────────────────
p8_return = m_p8["total_return"]
p8_sharpe = m_p8["sharpe_ratio"]
p8_maxdd  = m_p8["max_drawdown"]

mean_bull_ic_p7 = np.nanmean([bic for bic in
    [float("nan"), -0.1278, 0.1786, 0.0582, 0.0714]   # P-ML7 bull ICs from F14
    if not np.isnan(bic)])
mean_bull_ic_p8 = np.nanmean([ic for ic in bull_ics_p8 if not np.isnan(ic)])

ic_improved     = np.mean(ics_P8)  > np.mean(P_ML7_ICS)
bull_improved   = mean_bull_ic_p8  > mean_bull_ic_p7
sharpe_improved = p8_sharpe        > P_ML7["sharpe"]
dd_improved     = p8_maxdd         > P_ML7["maxdd"]   # less negative = better
n_vol_selected  = len(selected_vol_feats)

# Era trend: did IC grow from Era1 → Era4 for selected institutional features?
institutional_feats = [f for f in selected_vol_feats
                       if any(k in f for k in ["zscore", "cv", "signed", "trend"])]
era_trend_holds = False
if institutional_feats and era_ic_table:
    for feat in institutional_feats:
        ics = era_ic_table.get(feat, [])
        valid = [(i, abs(v)) for i, v in enumerate(ics) if not np.isnan(v)]
        if len(valid) >= 2:
            # Check if IC magnitude at Era4 > Era1
            first_abs = valid[0][1]
            last_abs  = valid[-1][1]
            if last_abs > first_abs:
                era_trend_holds = True
                break

print("=" * 75)
print("FINDING F15 — Volume Features (P-ML8) vs Momentum Baseline (P-ML7)")
print("=" * 75)
print()
print(f"Volume features screened: {len(VOL_CANDIDATES)}")
print(f"Selected (|IC_bull| > {IC_THRESHOLD}): {n_vol_selected}  → {selected_vol_feats}")
print(f"FEATURES_V3: {len(FEATURES_V2)} (P-ML7) + {n_vol_selected} (vol) = {len(FEATURES_V3)} features")
print()
print(f"{'Metric':<40} {'P-ML7 (16f)':>12} {'P-ML8 (+vol)':>13}")
print("-" * 67)
print(f"  {'Mean OOS IC':<38} {np.mean(P_ML7_ICS):>+12.4f} {np.mean(ics_P8):>+13.4f}")
print(f"  {'ICIR':<38} "
      f"{np.mean(P_ML7_ICS)/np.std(P_ML7_ICS):>+12.3f} "
      f"{np.mean(ics_P8)/np.std(ics_P8):>+13.3f}")
print(f"  {'Mean Bull IC':<38} {mean_bull_ic_p7:>+12.4f} {mean_bull_ic_p8:>+13.4f}")
print(f"  {'OOS Sharpe':<38} {P_ML7['sharpe']:>+12.3f} {p8_sharpe:>+13.3f}")
print(f"  {'OOS Return':<38} {P_ML7['return']*100:>+11.1f}% {p8_return*100:>+12.1f}%")
print(f"  {'Max Drawdown':<38} {P_ML7['maxdd']*100:>+11.1f}% {p8_maxdd*100:>+12.1f}%")
print()
print(f"IC improved?             {'YES' if ic_improved     else 'NO'}")
print(f"Mean bull IC improved?   {'YES' if bull_improved   else 'NO'}")
print(f"Sharpe improved?         {'YES' if sharpe_improved else 'NO'}")
print(f"MaxDD improved?          {'YES' if dd_improved     else 'NO'}")
print(f"Institutional era trend? {'YES — IC magnitude grows Era1→Era4' if era_trend_holds else 'NOT CONFIRMED'}")
print()

if sharpe_improved:
    verdict = (
        "VOLUME FEATURES ADD VALUE: Sharpe improves over P-ML7 momentum baseline. "
        "Volume confirms price moves and provides directional conviction signal beyond "
        "what price-based oscillators alone encode."
    )
elif n_vol_selected == 0:
    verdict = (
        "NO VOLUME FEATURES SELECTED: All 8 candidates failed the |IC_bull| > 0.01 "
        "IC screen. Volume signal is too weak at the daily horizon relative to the "
        "16 price-based features already in FEATURES_V2. Volume effects may be more "
        "informative at intraday resolution or with larger bull dataset."
    )
else:
    verdict = (
        f"MARGINAL VOLUME SIGNAL: {n_vol_selected} features passed IC screen but "
        "did not improve Sharpe over P-ML7. The volume signal is real but redundant "
        "with existing momentum / oscillator features in FEATURES_V2, or too weak to "
        "overcome the noise at binary position sizing."
    )

print(f"VERDICT: {verdict}")
print()
print("Institutional participation theory:")
if era_trend_holds:
    print("  SUPPORTED — volume IC grows from Era1 (retail) → Era4 (ETF), consistent with")
    print("  institutional footprints becoming more predictive as institutional share grows.")
else:
    print("  NOT CONFIRMED — IC does not consistently increase across eras. Volume patterns")
    print("  may not track institutional adoption clearly at daily granularity, or the 6yr")
    print("  dataset is too short to isolate the structural shift.")
print()
print("Next steps:")
if sharpe_improved:
    print("  - FEATURES_V3 becomes the new baseline for strategy integration (P-ML9)")
    print("  - P-ML9: Strategy integration — RegimeLGBMStrategy class")
    print("  - P-ML10: Risk overlay — position sizing + drawdown brake")
else:
    print("  - Retain FEATURES_V2 (P-ML7, 16 features) as current best feature set")
    print("  - Proceed to P-ML9: Strategy integration — RegimeLGBMStrategy class")
    print("  - Future: revisit volume at intraday (1h) frequency when hourly data added")

FINDING F15 — Volume Features (P-ML8) vs Momentum Baseline (P-ML7)

Volume features screened: 9
Selected (|IC_bull| > 0.01): 8  → ['vol_log_ratio_7d', 'vol_log_ratio_14d', 'vol_log_ratio_30d', 'vol_cv_14d', 'vol_zscore_30d', 'vol_signed_ratio_7d', 'vol_signed_ratio_14d', 'vol_price_corr_14d']
FEATURES_V3: 16 (P-ML7) + 8 (vol) = 24 features

Metric                                    P-ML7 (16f)  P-ML8 (+vol)
-------------------------------------------------------------------
  Mean OOS IC                                 +0.0740       +0.0463
  ICIR                                         +1.779        +0.747
  Mean Bull IC                                +0.0451       +0.1204
  OOS Sharpe                                   +1.261        +0.180
  OOS Return                                 +1997.6%        -43.2%
  Max Drawdown                                 -77.3%        -91.5%

IC improved?             NO
Mean bull IC improved?   YES
Sharpe improved?         NO
MaxDD improved?          NO